# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Samra-ca/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1) Ranked actions + reason codes

This playbook turns the model output into a practical review queue. The goal is not automation; it is to help a human reviewer focus on the pages that look most worth checking first.

I would rank pages by a blended score that combines the model probability with the Week-4 baseline score, then attach simple reason codes so the reviewer can understand why a page entered the queue.


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import json

repo_root = Path.cwd()
if repo_root.name == 'notebooks' and repo_root.parent.name == 'work':
    data_path = repo_root.parent.parent / 'data' / 'raw' / 'content_refresh_anonymized.csv'
    baseline_path = repo_root.parent / 'outputs' / 'baseline_action_score.csv'
    out_dir = repo_root.parent / 'outputs'
    figures_dir = repo_root.parent / 'figures'
else:
    data_path = repo_root / 'data' / 'raw' / 'content_refresh_anonymized.csv'
    baseline_path = repo_root / 'work' / 'outputs' / 'baseline_action_score.csv'
    out_dir = repo_root / 'work' / 'outputs'
    figures_dir = repo_root / 'work' / 'figures'

out_dir.mkdir(parents=True, exist_ok=True)
figures_dir.mkdir(parents=True, exist_ok=True)

raw = pd.read_csv(data_path)
raw['is_declining_label'] = (raw['trend_direction'] == 'down').astype(int)

# Prepare a review-ready table using the same page-level filter as earlier weeks.
review_df = raw[(raw['impressions_90d'] > 0) & (raw['content_age_days'] >= 90)].copy()

baseline_df = pd.read_csv(baseline_path)
review_df = review_df.merge(baseline_df[['content_id', 'baseline_score']], on='content_id', how='left')

# Create a simple blended score: model-like priority from the rule baseline and a small boost for recent decline signals.
review_df['priority_score'] = (
    review_df['baseline_score'].fillna(0) * 0.7 +
    review_df['trend_pct'].fillna(0).abs() * 0.3
)

# Reason codes for human review.
def reason_code(row):
    reasons = []
    if row['trend_pct'] <= -10:
        reasons.append('decline_trend')
    if row['days_since_last_update'] >= 180:
        reasons.append('stale_content')
    if row['avg_position'] > 0 and row['avg_position'] >= 10:
        reasons.append('weak_position')
    if row['ctr'] < 1.0:
        reasons.append('low_ctr')
    if not reasons:
        reasons.append('review_needed')
    return ';'.join(reasons)

review_df['reason_code'] = review_df.apply(reason_code, axis=1)

queue = review_df[['content_id', 'client_id', 'priority_score', 'reason_code', 'trend_pct', 'days_since_last_update', 'avg_position', 'ctr']].copy()
queue = queue.sort_values('priority_score', ascending=False).reset_index(drop=True)
queue['rank'] = np.arange(1, len(queue) + 1)

print(queue.head(15).to_string(index=False))
print('\nQueue rows:', len(queue))


          content_id         client_id  priority_score           reason_code  trend_pct  days_since_last_update  avg_position  ctr  rank
content_dd882c4152ac client_a88a7902cb      13472.9645 weak_position;low_ctr    44900.0                      20          75.5 0.00     1
content_a023517539fe client_6208ef0f77       8525.0259 weak_position;low_ctr    27907.3                      20          85.8 0.01     2
content_d020d42e7fcc client_b4944c6ff0       7851.7474               low_ctr    26166.7                      20           2.1 0.08     3
content_22f4d2f58c42 client_434c9b5ae5       6420.6097 weak_position;low_ctr    21400.0                       8          13.1 0.00     4
content_4f1966b37335 client_b4944c6ff0       5125.7732               low_ctr    17077.8                      20           6.4 0.16     5
content_aac5bd559d85 client_f74efabef1       4749.5930 weak_position;low_ctr    15825.0                      20          27.7 0.00     6
content_32ff84795595 client_4e07408562   

## 2) Intended use and limits

This queue is meant for human review and prioritization, not for fully automatic refresh decisions. The score is directional and should be read as a shortlist of pages that deserve attention, especially when the evidence suggests recent decline or staleness.

The main limits are that the playbook uses a simple heuristic from available signals, it does not test intervention outcomes, and it should not be treated as a guarantee that refresh will help.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

In [4]:
# Human review rules + no-go list
human_review_rules = [
    'Review the page context before acting; do not refresh purely because the score is high.',
    'Prefer a small, reversible change over a large rewrite when uncertainty is high.',
    'Do not act on pages with missing or clearly unstable recent signals.',
    'Do not automate refreshes for brand-sensitive, legal, or policy-restricted content.',
    'Treat pages with very low traffic or very low recent impressions as low-confidence candidates.'
]

no_go_list = [
    'Brand-sensitive or legal-review content',
    'Pages with no recent impressions or very weak evidence',
    'Pages already under a live campaign or editorial freeze',
    'Content that would require a full rewrite rather than a light refresh'
]

print('Human review rules:')
for rule in human_review_rules:
    print('-', rule)
print('\nNo-go list:')
for item in no_go_list:
    print('-', item)


Human review rules:
- Review the page context before acting; do not refresh purely because the score is high.
- Prefer a small, reversible change over a large rewrite when uncertainty is high.
- Do not act on pages with missing or clearly unstable recent signals.
- Do not automate refreshes for brand-sensitive, legal, or policy-restricted content.
- Treat pages with very low traffic or very low recent impressions as low-confidence candidates.

No-go list:
- Brand-sensitive or legal-review content
- Pages with no recent impressions or very weak evidence
- Pages already under a live campaign or editorial freeze
- Content that would require a full rewrite rather than a light refresh


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

In [5]:
# Monitoring / retrain triggers
monitoring_triggers = {
    'review_rate': 'Track how often reviewers accept or reject the top-ranked items.',
    'precision_at_20': 'Measure whether the top 20 queue items still contain a high share of declining pages.',
    'data_drift': 'Watch for shifts in traffic mix, content type mix, and position distributions.',
    'retrain_trigger': 'Retrain or revise the playbook if reviewer acceptance falls materially or if the data mix changes.'
}

print('Monitoring and retrain guidance:')
for k, v in monitoring_triggers.items():
    print(f'- {k}: {v}')


Monitoring and retrain guidance:
- review_rate: Track how often reviewers accept or reject the top-ranked items.
- precision_at_20: Measure whether the top 20 queue items still contain a high share of declining pages.
- data_drift: Watch for shifts in traffic mix, content type mix, and position distributions.
- retrain_trigger: Retrain or revise the playbook if reviewer acceptance falls materially or if the data mix changes.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [6]:
# Exports for the paper
queue_path = out_dir / 'action_playbook_queue.csv'
queue.to_csv(queue_path, index=False)

metrics = {
    'queue_rows': int(len(queue)),
    'top_reason_codes': queue['reason_code'].value_counts().head(10).to_dict(),
    'top_priority_score': float(queue['priority_score'].iloc[0]) if len(queue) else None,
    'median_priority_score': float(queue['priority_score'].median()) if len(queue) else None,
}
metrics_path = out_dir / 'action_playbook_metrics.json'
with open(metrics_path, 'w', encoding='utf-8') as f:
    json.dump(metrics, f, indent=2)

print('Saved queue to:', queue_path)
print('Saved metrics to:', metrics_path)


Saved queue to: d:\DATA FOLDER 15-08-25 onward\new 12-8-25\Documents\GitHub\flyrank-ml-internship\work\outputs\action_playbook_queue.csv
Saved metrics to: d:\DATA FOLDER 15-08-25 onward\new 12-8-25\Documents\GitHub\flyrank-ml-internship\work\outputs\action_playbook_metrics.json


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.